# TauGenNet — Tables & Figures

Interactive notebook for the TauGenNet paper.  Inference results are **cached to disk** after the first run, so all plotting cells execute instantly on repeat runs.

**Available figures**
| Cell | Figure | Notes |
|------|--------|-------|
| Setup | Config, ROI defs, helpers | Set `MODE` here |
| Load | Dataset + model | |
| Inference | Run synthesis or load cache | ~5–7 h first run; instant after |
| Fig 3 | Training loss curve | Instant |
| Fig A | Real / Generated / \|Diff\| — 6 subjects | ~30 s (DDIM 50 steps) |
| Metrics | SSIM · PSNR · MAE per-subject boxplots | From cache |
| Table I / Fig B | MRI ablation MSE bar chart | From cache |
| Table II / Fig C | NRMSE × plasma bin heatmap | From cache; ptau/combined only |
| Fig 4 | Plasma sweep (4 values) | ~2 min; ptau/combined only |
| Fig 5 | ROI distributions by plasma bin | From cache; ptau/combined only |
| SUVR MAE | Regional \|SUVR\_gen − SUVR\_real\| | From cache |
| ROI MSE | Per-subject ROI MSE — all 6 model runs | Loads cache for all 3 modes |


## Setup — imports, config, helpers

In [ ]:
import os, sys, pickle
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import torch
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display
from tqdm.notebook import tqdm
from skimage.metrics import structural_similarity as ssim_fn

from src.config import DEVICE, VOL_SHAPE
from src.diffusion import DiffusionSchedule
from src.inference import load_models, synthesize_tau_pet, synthesize_no_mri

# ── Config ─────────────────────────────────────────────────────────────────────
MODE    = 'combined'   # 'atrophy' | 'ptau217' | 'combined'
N_STEPS = 500          # DDPM steps for full-quality inference (cached records)
N_FAST  = 50           # DDIM steps for quick on-demand figures

CKPT       = f'results/checkpoints/{MODE}_paper/diff_{MODE}.pt'
FIG_DIR    = f'results/figures/{MODE}/paper'
CACHE_DIR  = 'results/records'
CACHE_FILE = os.path.join(CACHE_DIR, f'{MODE}_paper.pkl')
HAS_PTAU   = MODE in ('ptau217', 'combined')
os.makedirs(FIG_DIR,  exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
print(f'Mode : {MODE}')
print(f'Cache: {CACHE_FILE}  ({"exists" if os.path.exists(CACHE_FILE) else "will be created on inference run"})')

# ── ROI definitions (approx bounding boxes as volume fractions) ────────────────
H, W, D = VOL_SHAPE
ROI_DEFS = {
    'Parahippocampal':   (0.40, 0.55, 0.35, 0.65, 0.30, 0.55),
    'Fusiform':          (0.45, 0.60, 0.30, 0.70, 0.25, 0.50),
    'Inferior Temporal': (0.30, 0.55, 0.25, 0.75, 0.20, 0.55),
    'Hippocampus':       (0.42, 0.52, 0.40, 0.60, 0.35, 0.50),
    'Post. Cingulate':   (0.40, 0.55, 0.38, 0.62, 0.50, 0.70),
    'Entorhinal':        (0.43, 0.55, 0.38, 0.62, 0.28, 0.45),
}
PLASMA_BINS = [(0, 2), (2, 4), (4, 6), (6, 8), (10, float('inf'))]
BIN_LABELS  = ['0–2', '2–4', '4–6', '6–8', '10+']

def roi_mean(vol, roi):
    y0, y1, x0, x1, z0, z1 = roi
    return float(vol[int(y0*H):int(y1*H), int(x0*W):int(x1*W), int(z0*D):int(z1*D)].mean())

def get_ptau(cond_i):
    return cond_i[86].item() if MODE == 'combined' else cond_i.item()

def bin_label(ptau_val):
    for (lo, hi), lbl in zip(PLASMA_BINS, BIN_LABELS):
        if lo <= ptau_val < hi:
            return lbl
    return None


## Load dataset and model

In [ ]:
if MODE == 'combined':
    from src import dataset_combined as _dc
    _, _, test_ds, _, _, test_loader = _dc.build_dataloaders(mode=MODE)
else:
    from src.dataset import build_dataloaders
    _, _, test_ds, _, _, test_loader = build_dataloaders(mode=MODE)

ae, unet, conditioner, latent_std, diff_losses = load_models(CKPT, MODE)
encode_cond = conditioner.encode
schedule    = DiffusionSchedule()
print(f'Test set : {len(test_ds)} subjects')
print(f'Epochs   : {len(diff_losses)}  |  final loss: {diff_losses[-1]:.6f}' if diff_losses else 'No loss history')


## Inference (run once, cached)

Set `FORCE_RERUN = True` to regenerate the cache.  On first run this takes ~5–7 h on GPU; subsequent runs load from pickle in seconds.

Records stored: MSE, PSNR, MAE, SSIM, per-ROI SUVR means (real/full/only), per-ROI MAE.
Volumes are **not** cached — they are regenerated on demand for Fig A.


In [ ]:
FORCE_RERUN = False

if not FORCE_RERUN and os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, 'rb') as f:
        records = pickle.load(f)
    print(f'Loaded {len(records)} records from {CACHE_FILE}')
else:
    print(f'Running inference on {len(test_ds)} subjects ({N_STEPS} DDPM steps) ...')
    records = []
    torch.manual_seed(0)
    for pet_b, mri_b, cond_b in tqdm(test_loader, desc='Synthesising'):
        for j in range(pet_b.shape[0]):
            real_np  = pet_b[j].squeeze().numpy()
            gen_full = synthesize_tau_pet(
                mri_b[j:j+1], cond_b[j], ae, unet, schedule,
                encode_cond, latent_std, device=DEVICE, n_steps=N_STEPS, sampler='ddpm'
            ).squeeze().numpy()
            gen_only = synthesize_no_mri(
                mri_b[j:j+1], cond_b[j], ae, unet, schedule,
                encode_cond, latent_std, device=DEVICE, n_steps=N_STEPS
            ).squeeze().numpy()

            mse_val = float(np.mean((gen_full - real_np) ** 2))
            rec = {
                'mse':           mse_val,
                'psnr':          float(10 * np.log10(1.0 / (mse_val + 1e-10))),
                'mae':           float(np.mean(np.abs(gen_full - real_np))),
                'ssim':          float(ssim_fn(real_np, gen_full, data_range=1.0)),
                'real_vol_mean': float(real_np.mean()),   # needed for whole-volume NRMSE
            }
            if HAS_PTAU:
                rec['ptau'] = get_ptau(cond_b[j])
            for rname, roi in ROI_DEFS.items():
                rec[f'real_{rname}'] = roi_mean(real_np,  roi)
                rec[f'full_{rname}'] = roi_mean(gen_full, roi)
                rec[f'only_{rname}'] = roi_mean(gen_only, roi)
                rec[f'roi_mae_{rname}'] = abs(rec[f'full_{rname}'] - rec[f'real_{rname}'])
            records.append(rec)

    with open(CACHE_FILE, 'wb') as f:
        pickle.dump(records, f)
    print(f'Saved {len(records)} records → {CACHE_FILE}')

# Summary
print(f"Mean SSIM : {np.mean([r['ssim'] for r in records]):.4f} ± {np.std([r['ssim'] for r in records]):.4f}")
print(f"Mean PSNR : {np.mean([r['psnr'] for r in records]):.2f} ± {np.std([r['psnr'] for r in records]):.2f} dB")
print(f"Mean MSE  : {np.mean([r['mse']  for r in records]):.6f} ± {np.std([r['mse'] for r in records]):.6f}")
print(f"Mean MAE  : {np.mean([r['mae']  for r in records]):.4f} ± {np.std([r['mae'] for r in records]):.4f}")


## Fig 3 — Training loss curve

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
if diff_losses:
    ax.plot(diff_losses, color='#4C72B0', linewidth=1.3, label='Train MSE')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Diffusion MSE')
    ax.set_title(f'Training loss — {MODE} model')
    ax.legend(fontsize=9)
else:
    ax.text(0.5, 0.5, 'No loss history in checkpoint', ha='center', va='center',
            transform=ax.transAxes, fontsize=12, color='gray')
plt.tight_layout()
_path = os.path.join(FIG_DIR, 'fig3_training_curve.pdf')
plt.savefig(_path, bbox_inches='tight', dpi=200); plt.show(); print(f'Saved → {_path}')


## Fig A — Multi-subject comparison (Real | Generated | |Diff|)

Runs **DDIM at `N_FAST` steps** on the first 6 test subjects — takes ~30 s on GPU.


In [ ]:
N_SUBJ = 6
torch.manual_seed(0)
subjects, per_ssim = [], []
for pet_b, mri_b, cond_b in test_loader:
    for j in range(pet_b.shape[0]):
        if len(subjects) >= N_SUBJ:
            break
        real_np = pet_b[j].squeeze().numpy()
        gen_np  = synthesize_tau_pet(
            mri_b[j:j+1], cond_b[j], ae, unet, schedule,
            encode_cond, latent_std, device=DEVICE, n_steps=N_FAST, sampler='ddim'
        ).squeeze().numpy()
        subjects.append((real_np, gen_np))
        per_ssim.append(ssim_fn(real_np, gen_np, data_range=1.0))
    if len(subjects) >= N_SUBJ:
        break

fig, axes = plt.subplots(N_SUBJ, 3, figsize=(9, N_SUBJ * 2.2))
for row, (real_np, gen_np) in enumerate(subjects):
    mid    = real_np.shape[2] // 2
    real_s = real_np[:, :, mid]; gen_s = gen_np[:, :, mid]; diff_s = np.abs(real_s - gen_s)
    axes[row, 0].imshow(real_s, cmap='hot', vmin=0, vmax=1)
    axes[row, 1].imshow(gen_s,  cmap='hot', vmin=0, vmax=1)
    im = axes[row, 2].imshow(diff_s, cmap='coolwarm', vmin=0, vmax=0.3)
    axes[row, 0].set_ylabel(f'S{row+1}  SSIM={per_ssim[row]:.3f}', fontsize=8)
    for ax in axes[row]: ax.axis('off')
axes[0,0].set_title('Real tau PET',      fontsize=11, fontweight='bold')
axes[0,1].set_title('Generated tau PET', fontsize=11, fontweight='bold')
axes[0,2].set_title('|Difference|',      fontsize=11, fontweight='bold')
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
fig.colorbar(im, cax=cbar_ax, label='Abs. error')
fig.suptitle(f'Tau PET synthesis — {MODE}  |  mean SSIM = {np.mean(per_ssim):.3f} ± {np.std(per_ssim):.3f}',
             fontsize=11, y=1.01)
plt.subplots_adjust(left=0.12, right=0.90, hspace=0.05, wspace=0.05)
_path = os.path.join(FIG_DIR, 'fig_a_multi_subject.pdf')
plt.savefig(_path, bbox_inches='tight', dpi=200); plt.show(); print(f'Saved → {_path}')


## Metrics summary — SSIM, PSNR, MAE (per-subject distributions)

In [ ]:
metrics_cfg = [('ssim', 'SSIM'), ('psnr', 'PSNR (dB)'), ('mae', 'MAE')]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (key, lbl) in zip(axes, metrics_cfg):
    vals = [r[key] for r in records]
    bp = ax.boxplot([vals], patch_artist=True, widths=0.5,
                    medianprops=dict(color='black', linewidth=1.5))
    bp['boxes'][0].set_facecolor('#4C72B0'); bp['boxes'][0].set_alpha(0.8)
    ax.set_xticklabels([MODE], fontsize=10)
    ax.set_ylabel(lbl, fontsize=9)
    ax.set_title(f'{lbl}\nmean={np.mean(vals):.3f}  std={np.std(vals):.3f}', fontsize=9)
fig.suptitle(f'Per-subject metric distributions — {MODE}', fontsize=11)
plt.tight_layout()
_path = os.path.join(FIG_DIR, 'metrics_distributions.pdf')
plt.savefig(_path, bbox_inches='tight', dpi=200); plt.show(); print(f'Saved → {_path}')


## NRMSE — Normalized Root Mean Square Error

Two complementary views, both dataset-scale-invariant:

- **Whole-volume NRMSE** per subject: `sqrt(MSE) / mean(real_vol)` — normalises by each subject's mean PET signal. Useful for cross-dataset comparison since it removes dependence on the absolute SUVR scale.
- **Per-ROI NRMSE** (group-level): `|mean_real_roi − mean_gen_roi| / mean_real_roi` — same normalisation applied per region. Equivalent to the values in Table II but aggregated over all subjects rather than stratified by plasma bin.

> **Note:** `real_vol_mean` is stored in the cache only when inference is run from this notebook. Records loaded from the SLURM job cache fall back to estimating the normaliser from the stored ROI means.


In [ ]:
roi_names = list(ROI_DEFS.keys())

# ── Whole-volume NRMSE per subject ────────────────────────────────────────────
# Uses real_vol_mean if available; falls back to mean of stored ROI means.
def _normaliser(rec):
    if 'real_vol_mean' in rec:
        return rec['real_vol_mean']
    return np.mean([rec[f'real_{rn}'] for rn in roi_names])

nrmse_per_subj = [np.sqrt(r['mse']) / (_normaliser(r) + 1e-10) for r in records]
print(f'Whole-volume NRMSE : {np.mean(nrmse_per_subj):.4f} ± {np.std(nrmse_per_subj):.4f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: per-subject NRMSE distribution
ax = axes[0]
bp = ax.boxplot([nrmse_per_subj], patch_artist=True, widths=0.5,
                medianprops=dict(color='black', linewidth=1.5))
bp['boxes'][0].set_facecolor('#4C72B0'); bp['boxes'][0].set_alpha(0.8)
ax.set_xticklabels([MODE], fontsize=10)
ax.set_ylabel('NRMSE  (sqrt(MSE) / mean_real)', fontsize=9)
ax.set_title(f'Whole-volume NRMSE\nmean={np.mean(nrmse_per_subj):.4f}  std={np.std(nrmse_per_subj):.4f}',
             fontsize=9)

# Right: per-ROI group-level NRMSE bar chart
ax = axes[1]
roi_nrmse_means, roi_nrmse_stds = [], []
for rn in roi_names:
    real_mu = np.mean([r[f'real_{rn}'] for r in records])
    gen_mu  = np.mean([r[f'full_{rn}'] for r in records])
    per_subj = [abs(r[f'real_{rn}'] - r[f'full_{rn}']) / (r[f'real_{rn}'] + 1e-10)
                for r in records]
    roi_nrmse_means.append(np.mean(per_subj))
    roi_nrmse_stds.append(np.std(per_subj))

x = np.arange(len(roi_names))
bars = ax.bar(x, roi_nrmse_means, 0.6, color='#DD8452', alpha=0.85,
              yerr=roi_nrmse_stds, capsize=4, error_kw=dict(linewidth=1.2))
for bar, m in zip(bars, roi_nrmse_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + bar.get_height()*0.05,
            f'{m:.3f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(roi_names, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Per-subject NRMSE  (mean ± std)', fontsize=9)
ax.set_title(f'Per-ROI NRMSE — {MODE} + MRI', fontsize=10, fontweight='bold')

fig.suptitle(f'NRMSE — {MODE}', fontsize=11)
plt.tight_layout()
_path = os.path.join(FIG_DIR, 'nrmse.pdf')
plt.savefig(_path, bbox_inches='tight', dpi=200); plt.show(); print(f'Saved → {_path}')

# ── Summary table ─────────────────────────────────────────────────────────────
nrmse_table = pd.DataFrame({
    'Mean NRMSE': roi_nrmse_means,
    'Std NRMSE':  roi_nrmse_stds,
}, index=roi_names)
nrmse_table.loc['Whole-volume'] = [np.mean(nrmse_per_subj), np.std(nrmse_per_subj)]
display(nrmse_table.style
    .format('{:.4f}')
    .background_gradient(cmap='YlOrRd', subset=['Mean NRMSE'])
    .set_caption(f'NRMSE summary — {MODE}'))


## Table I — MRI ablation  |  Fig B — bar chart

In [ ]:
label_map = {
    'atrophy':  ('MRI + Atrophy',          'Atrophy only'),
    'ptau217':  ('MRI + Plasma',           'Plasma only'),
    'combined': ('MRI + Atrophy + Plasma', 'Atrophy + Plasma only'),
}
lbl_full, lbl_only = label_map[MODE]

# Build Table I
rows = {}
for lbl, key_prefix in [(lbl_full, 'full'), (lbl_only, 'only')]:
    rows[lbl] = {}
    for rname in ROI_DEFS:
        real_mu = np.mean([r[f'real_{rname}'] for r in records])
        gen_mu  = np.mean([r[f'{key_prefix}_{rname}'] for r in records])
        rows[lbl][rname] = round((real_mu - gen_mu) ** 2, 6)

table1 = pd.DataFrame(rows).T
display(table1.style
    .format('{:.6f}')
    .background_gradient(cmap='YlOrRd', axis=None)
    .set_caption(f'Table I — Group-level MSE by ROI ({MODE})'))

# Fig B
roi_names = list(ROI_DEFS.keys())
x = np.arange(len(roi_names)); w = 0.35
fig, ax = plt.subplots(figsize=(10, 4))
b1 = ax.bar(x - w/2, [rows[lbl_only][r] for r in roi_names], w,
            label=lbl_only, color='#C44E52', alpha=0.85)
b2 = ax.bar(x + w/2, [rows[lbl_full][r] for r in roi_names], w,
            label=lbl_full,  color='#4C72B0', alpha=0.85)
for bars in (b1, b2):
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h*1.03, f'{h:.4f}',
                ha='center', va='bottom', fontsize=6.5, rotation=45)
ax.set_xticks(x); ax.set_xticklabels(roi_names, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Group-level MSE'); ax.set_title(f'Table I — MRI Ablation ({MODE})')
ax.legend(fontsize=9); plt.tight_layout()
_path = os.path.join(FIG_DIR, 'fig_b_table1_ablation.pdf')
plt.savefig(_path, bbox_inches='tight', dpi=200); plt.show(); print(f'Saved → {_path}')


## Table II — NRMSE by plasma bin  |  Fig C — heatmap

*Skipped automatically for `atrophy` mode.*


In [ ]:
if not HAS_PTAU:
    print(f'Table II / Fig C: skipped (mode={MODE} has no plasma conditioning).')
else:
    bin_real = {l: {r: [] for r in ROI_DEFS} for l in BIN_LABELS}
    bin_gen  = {l: {r: [] for r in ROI_DEFS} for l in BIN_LABELS}
    bin_n    = {l: 0 for l in BIN_LABELS}
    for rec in records:
        lbl = bin_label(rec.get('ptau', -1))
        if lbl is None: continue
        for r in ROI_DEFS:
            bin_real[lbl][r].append(rec[f'real_{r}'])
            bin_gen[lbl][r].append(rec[f'full_{r}'])
        bin_n[lbl] += 1

    # Table II
    nrmse_data = {}
    for lbl in BIN_LABELS:
        nrmse_data[lbl] = {}
        for r in ROI_DEFS:
            rv, gv = bin_real[lbl][r], bin_gen[lbl][r]
            nrmse_data[lbl][r] = abs(np.mean(rv) - np.mean(gv)) / (np.mean(rv) + 1e-8) if rv else float('nan')
    table2 = pd.DataFrame(nrmse_data).T
    display(table2.style
        .format('{:.4f}', na_rep='—')
        .background_gradient(cmap='YlOrRd', axis=None)
        .set_caption(f'Table II — NRMSE by plasma bin × ROI ({MODE})'))
    for lbl in BIN_LABELS:
        print(f'  {lbl} pg/mL : n={bin_n[lbl]}')

    # Fig C
    data = table2.values.astype(float)
    mask = np.isnan(data)
    fig, ax = plt.subplots(figsize=(10, 3.5))
    im = ax.imshow(np.where(mask, 0, data), cmap='YlOrRd', aspect='auto',
                   vmin=0, vmax=float(np.nanmax(data)) if not np.all(mask) else 1.0)
    for i, lbl in enumerate(BIN_LABELS):
        for j, rname in enumerate(list(ROI_DEFS.keys())):
            txt = '—' if mask[i,j] else f'{data[i,j]:.3f}'
            ax.text(j, i, txt, ha='center', va='center', fontsize=8,
                    color='gray' if mask[i,j] else 'black')
        if bin_n[lbl] > 0:
            ax.text(-0.55, i, f'n={bin_n[lbl]}', ha='right', va='center', fontsize=7, color='dimgray')
    ax.set_xticks(range(len(ROI_DEFS))); ax.set_xticklabels(list(ROI_DEFS.keys()), rotation=20, ha='right', fontsize=9)
    ax.set_yticks(range(len(BIN_LABELS))); ax.set_yticklabels([f'{l} pg/mL' for l in BIN_LABELS], fontsize=9)
    fig.colorbar(im, ax=ax, label='NRMSE')
    ax.set_title(f'Table II — NRMSE by plasma bin × ROI ({MODE})')
    plt.tight_layout()
    _path = os.path.join(FIG_DIR, 'fig_c_table2_nrmse.pdf')
    plt.savefig(_path, bbox_inches='tight', dpi=200); plt.show(); print(f'Saved → {_path}')


## Fig 4 — Plasma sweep

Fixed MRI; tau PET generated at 4 plasma values.  Runs DDIM at `N_FAST` steps (~2 min on GPU).
*Skipped automatically for `atrophy` mode.*


In [ ]:
PLASMA_SWEEP = [0.65, 3.65, 6.65, 10.65]

if not HAS_PTAU:
    print(f'Fig 4 skipped (mode={MODE}).')
else:
    # Grab first test subject
    for pet_b, mri_b, cond_b in test_loader:
        first_mri  = mri_b[0:1]
        first_cond = cond_b[0].clone()
        break

    def synth_at_plasma(mri_t, base_cond, pv):
        cond = base_cond.clone()
        if MODE == 'combined': cond[86] = pv
        else: cond = torch.tensor([pv], dtype=torch.float32)
        return synthesize_tau_pet(mri_t, cond, ae, unet, schedule,
                                  encode_cond, latent_std, device=DEVICE,
                                  n_steps=N_FAST, sampler='ddim').squeeze().numpy()

    torch.manual_seed(0)
    print(f'Synthesising at {len(PLASMA_SWEEP)} plasma values ({N_FAST} DDIM steps)...')
    vols = [synth_at_plasma(first_mri, first_cond, pv) for pv in PLASMA_SWEEP]

    view_lbls = ['Axial', 'Sagittal', 'Coronal']
    fig, axes = plt.subplots(3, len(PLASMA_SWEEP), figsize=(len(PLASMA_SWEEP)*2.8, 3*2.5))
    for col, (vol, pv) in enumerate(zip(vols, PLASMA_SWEEP)):
        h2, w2, d2 = vol.shape
        slices = [vol[:, :, d2//2], vol[:, w2//2, :], vol[h2//2, :, :]]
        for row, sl in enumerate(slices):
            im = axes[row, col].imshow(sl, cmap='hot', vmin=0, vmax=1)
            axes[row, col].axis('off')
        axes[0, col].set_title(f'Plasma {pv}', fontsize=10, fontweight='bold')
    for row, lbl in enumerate(view_lbls):
        axes[row, 0].set_ylabel(lbl, fontsize=9)
    cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
    fig.colorbar(im, cax=cbar_ax, label='SUVR (norm.)')
    fig.suptitle(f'Generated tau PET across plasma values — {MODE}', fontsize=11, y=1.01)
    plt.subplots_adjust(left=0.10, right=0.90, hspace=0.05, wspace=0.05)
    _path = os.path.join(FIG_DIR, 'fig4_plasma_sweep.pdf')
    plt.savefig(_path, bbox_inches='tight', dpi=200); plt.show(); print(f'Saved → {_path}')


## Fig 5 — ROI distributions by plasma bin

Real (grouped by subject plasma) vs. generated (at fixed plasma values).
Uses cached real distributions; generates new volumes for the 4 sweep values (~10 min on GPU).
*Skipped automatically for `atrophy` mode.*


In [ ]:
if not HAS_PTAU:
    print(f'Fig 5 skipped (mode={MODE}).')
else:
    # Real distributions from cached records
    real_bins = {l: {r: [] for r in ROI_DEFS} for l in BIN_LABELS}
    for rec in records:
        lbl = bin_label(rec.get('ptau', -1))
        if lbl is None: continue
        for r in ROI_DEFS:
            real_bins[lbl][r].append(rec[f'real_{r}'])

    # Generated distributions across all test subjects at fixed plasma values
    gen_bins = {pv: {r: [] for r in ROI_DEFS} for pv in PLASMA_SWEEP}
    torch.manual_seed(0)
    for _, mri_b, cond_b in tqdm(test_loader, desc='Fig 5 sweep'):
        for j in range(mri_b.shape[0]):
            for pv in PLASMA_SWEEP:
                vol = synth_at_plasma(mri_b[j:j+1], cond_b[j], pv)
                for r, roi in ROI_DEFS.items():
                    gen_bins[pv][r].append(roi_mean(vol, roi))

    real_color, gen_color = '#E8A0A0', '#A0C4E8'
    x_labels = []
    for rl, pv in zip(BIN_LABELS, PLASMA_SWEEP):
        x_labels += [rl, f'Gen {pv}']

    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    for ax, rname in zip(axes.flatten(), list(ROI_DEFS.keys())):
        box_data, box_colors = [], []
        for rl, pv in zip(BIN_LABELS, PLASMA_SWEEP):
            box_data.append(real_bins[rl][rname]); box_colors.append(real_color)
            box_data.append(gen_bins[pv][rname]);  box_colors.append(gen_color)
        bp = ax.boxplot(box_data, patch_artist=True, widths=0.6,
                        medianprops=dict(color='black', linewidth=1.2))
        for patch, color in zip(bp['boxes'], box_colors):
            patch.set_facecolor(color); patch.set_alpha(0.85)
        for i, data in enumerate(box_data):
            if data:
                ax.hlines(np.mean(data), i+0.7, i+1.3, colors='red', linestyles='dashed', linewidth=1.0)
        ax.set_xticks(range(1, len(x_labels)+1))
        ax.set_xticklabels(x_labels, rotation=30, ha='right', fontsize=7)
        ax.set_ylabel('Avg. PET (region)', fontsize=8); ax.set_title(rname, fontsize=10, fontweight='bold')
    fig.legend(handles=[Patch(facecolor=real_color, label='Real'), Patch(facecolor=gen_color, label='Generated')],
               loc='upper right', fontsize=9)
    fig.suptitle(f'PET distributions by plasma bin — {MODE}', fontsize=12, y=1.01)
    plt.tight_layout()
    _path = os.path.join(FIG_DIR, 'fig5_roi_boxplots.pdf')
    plt.savefig(_path, bbox_inches='tight', dpi=200); plt.show(); print(f'Saved → {_path}')


## Regional SUVR MAE — |generated − real| per ROI

In [ ]:
roi_names = list(ROI_DEFS.keys())
mae_means = [np.mean([r[f'roi_mae_{rn}'] for r in records]) for rn in roi_names]
mae_stds  = [np.std( [r[f'roi_mae_{rn}'] for r in records]) for rn in roi_names]

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(roi_names))
bars = ax.bar(x, mae_means, 0.6, color='#4C72B0', alpha=0.85,
              yerr=mae_stds, capsize=4, error_kw=dict(linewidth=1.2))
for bar, m, s in zip(bars, mae_means, mae_stds):
    ax.text(bar.get_x() + bar.get_width()/2, m+s+0.001, f'{m:.4f}',
            ha='center', va='bottom', fontsize=8, rotation=45)
ax.set_xticks(x); ax.set_xticklabels(roi_names, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Mean |SUVR gen − real|'); ax.set_title(f'Regional SUVR MAE — {MODE} + MRI')
plt.tight_layout()
_path = os.path.join(FIG_DIR, 'roi_suvr_mae.pdf')
plt.savefig(_path, bbox_inches='tight', dpi=200); plt.show(); print(f'Saved → {_path}')


## ROI MSE boxplots — all 6 model runs

Loads cached records for all three modes. Run the inference cell for each mode first (changing `MODE` in Setup and re-running through the inference cell).


In [ ]:
all_modes = ['atrophy', 'ptau217', 'combined']
all_records = {}
for m in all_modes:
    p = os.path.join(CACHE_DIR, f'{m}_paper.pkl')
    if os.path.exists(p):
        with open(p, 'rb') as f:
            all_records[m] = pickle.load(f)
        print(f'  {m}: {len(all_records[m])} records')
    else:
        print(f'  {m}: cache not found ({p}) — run inference for this mode first')

run_labels = ['Atrophy\n+MRI', 'Atrophy\nonly', 'Plasma\n+MRI',
              'Plasma\nonly', 'Both\n+MRI',   'Both\nonly']
colors = ['#4C72B0','#A8C4E8', '#DD8452','#F5C49A', '#55A868','#A8D4B4']

run_specs = [
    ('atrophy',  'full'), ('atrophy',  'only'),
    ('ptau217',  'full'), ('ptau217',  'only'),
    ('combined', 'full'), ('combined', 'only'),
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, rname in zip(axes.flatten(), list(ROI_DEFS.keys())):
    box_data = []
    for m, key in run_specs:
        recs = all_records.get(m, [])
        if key == 'full':
            box_data.append([(r[f'real_{rname}'] - r[f'full_{rname}'])**2 for r in recs])
        else:
            box_data.append([(r[f'real_{rname}'] - r[f'only_{rname}'])**2 for r in recs])
    valid = [d for d in box_data if d]
    if not valid:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        continue
    bp = ax.boxplot(box_data, patch_artist=True, widths=0.6,
                    medianprops=dict(color='black', linewidth=1.5))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color); patch.set_alpha(0.85)
    ax.set_xticks(range(1, len(run_labels)+1))
    ax.set_xticklabels(run_labels, fontsize=8)
    ax.set_ylabel('(SUVR real − gen)²', fontsize=8)
    ax.set_title(rname, fontsize=10, fontweight='bold')

legend_elements = [Patch(facecolor=c, label=l) for c, l in zip(colors, run_labels)]
fig.legend(handles=legend_elements, loc='upper right', fontsize=8, ncol=2)
fig.suptitle('Per-subject ROI MSE — all 6 model runs', fontsize=12, y=1.01)
plt.tight_layout()
_path = 'results/figures/roi_mse_boxplots.pdf'
plt.savefig(_path, bbox_inches='tight', dpi=200); plt.show(); print(f'Saved → {_path}')
